# Steam Games — Data Cleaning, Normalization & Qwen 7B Tag Mapping

Notebook này gồm 2 phần chính:

1. **Cleaning & imputation**: xử lý dữ liệu thiếu, chuẩn hóa các cột cơ bản và lưu dataset trung gian.
2. **Normalization & tag mapping**: chuẩn hóa price, OS, RAM, CPU, tags; sau đó dùng **Qwen 7B 4-bit** để map những tag hiếm/khó về bộ tag chuẩn.

Mục tiêu cuối cùng là tạo ra dữ liệu sạch hơn, đặc biệt là cột tags đã được chuẩn hóa để phục vụ bước phân tích hoặc recommendation system sau này.


# Phần 1: PreprocessingNULLValue(1).ipynb

## Thiết lập đường dẫn project

Cell này tạo các biến đường dẫn dùng chung cho notebook:

- `PROJECT_ROOT`: thư mục gốc của project, tự nhận biết nếu notebook nằm trong folder `notebooks`.
- `DATA_DIR`, `RAW_DIR`, `INTERIM_DIR`, `PROCESSED_DIR`: các thư mục dữ liệu theo chuẩn pipeline.
- `RAW_PATH`, `INTERIM_CLEAN_PATH`, `PROCESSED_PATH`, `TAG_MAPPING_PATH`: các file input/output chính.

Việc gom đường dẫn vào một cell giúp notebook dễ chạy lại trên nhiều môi trường như local hoặc Kaggle.


In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_PATH = RAW_DIR / "SteamGames.csv"
PROCESSED_PATH = PROCESSED_DIR / "SteamGames_cleaned.csv"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


# Preprocessing

Bản gộp và chỉnh lại pipeline xử lý dữ liệu.



## Import thư viện xử lý dữ liệu

Cell này import các thư viện cơ bản:

- `pandas`, `numpy`: xử lý bảng dữ liệu.
- `matplotlib`, `seaborn`: trực quan hóa và kiểm tra phân phối dữ liệu.
- `re`: xử lý text bằng regular expression.

Các thư viện này được dùng xuyên suốt phần cleaning và normalization.


In [2]:
import numpy as np 
import matplotlib.pyplot as plt 
import pandas as pd
import seaborn as sns
import re


## Đọc dữ liệu SteamGames.csv

Cell này đọc file dữ liệu gốc từ Kaggle nếu tồn tại. Nếu không, có thể dùng đường dẫn project đã khai báo ở trên.

Sau khi đọc, biến chính của notebook là `df`. Tất cả bước làm sạch tiếp theo sẽ thao tác trên DataFrame này.


In [ ]:
# Đường dẫn Kaggle hiện tại của bạn. Nếu chạy theo project structure, dùng RAW_PATH.
kaggle_path = Path('/kaggle/input/datasets/newnguyn/steamgame/SteamGames.csv')
df = pd.read_csv(kaggle_path if kaggle_path.exists() else RAW_PATH)


### Do vấn đề về việc cào không liên tục (cào trong nhiều đợt ngắt quãng) do trang web chặn, nên bị trùng xếp hạng về độ phổ biến, giờ ta phải xử lí lại đễ rank được liên tục


## Kiểm tra rank bị trùng

Do dữ liệu được crawl nhiều đợt, một số game có thể bị trùng `Rank`, ví dụ nhiều dòng cùng `Rank = 1`.

Cell này hiển thị các dòng có `Rank = 1` để xác nhận vấn đề trước khi sửa.


In [4]:
display(df[df['Rank'] == 1])

,Appid,Name,Type,ReleaseDate,Genres,Developers,Publishers,Description,price,Thumbnail,Tags,ReviewScore,PositiveReview,NegativeReview,OsRequirement,MemoryRequirement,CpuRequirement,Rank
0,3764200,Resident Evil Requiem,game,"Feb 26, 2026","Action,Adventure","CAPCOM Co., Ltd.","CAPCOM Co., Ltd.",Resident Evil Requiem Deluxe Edition Resident ...,69.99$,https://shared.akamai.steamstatic.com/store_it...,NaN,9,23715,793,NaN,NaN,NaN,1
13212,4418930,The Bazaar - Karnok,dlc,"Mar 4, 2026",Strategy,Tempo,NaN,The Monstrous Hunter Expansion adds over 120 n...,19.99$,https://shared.akamai.steamstatic.com/store_it...,"Strategy,Auto Battler,Deckbuilding,Card Battle...",0,7,0,Windows 10 (64-bit),3 GB RAM,AMD 2.4GHz 2-Core or Intel i3 2.4GHz 2-Core,1


## Đánh lại Rank liên tục

Cell này tạo lại cột `Rank` bằng `np.arange(1, len(df) + 1)`.

Ý nghĩa: sau khi dữ liệu đã được sắp theo thứ tự popularity ban đầu, ta gán lại rank liên tục từ 1 đến số dòng cuối cùng để loại bỏ rank trùng.


In [5]:
df['Rank'] = np.arange(1, len(df) + 1)

## Kiểm tra lại Rank sau khi sửa

Cell này hiển thị lại dòng có `Rank = 1` để đảm bảo sau khi đánh lại rank, chỉ còn một game đứng đầu.


In [6]:
display(df[df['Rank'] == 1])

,Appid,Name,Type,ReleaseDate,Genres,Developers,Publishers,Description,price,Thumbnail,Tags,ReviewScore,PositiveReview,NegativeReview,OsRequirement,MemoryRequirement,CpuRequirement,Rank
0,3764200,Resident Evil Requiem,game,"Feb 26, 2026","Action,Adventure","CAPCOM Co., Ltd.","CAPCOM Co., Ltd.",Resident Evil Requiem Deluxe Edition Resident ...,69.99$,https://shared.akamai.steamstatic.com/store_it...,NaN,9,23715,793,NaN,NaN,NaN,1


In [7]:
# Vậy ta đã xử lí xong việc bị trùng lại rank do việc cào ngắt quãng

### Xử lí dữ liệu bị NULL 


## Kiểm tra số lượng giá trị thiếu ban đầu

Cell này dùng `df.isnull().sum()` để xem mỗi cột đang có bao nhiêu giá trị null.

Kết quả này giúp quyết định cột nào nên xóa dòng, cột nào nên fill bằng giá trị thay thế.


In [8]:
df.isnull().sum()

Appid                   0
Name                    0
Type                    0
ReleaseDate            14
Genres                413
Developers             26
Publishers           1045
Description            52
price                   0
Thumbnail               0
Tags                 1614
ReviewScore             0
PositiveReview          0
NegativeReview          0
OsRequirement        2937
MemoryRequirement    3984
CpuRequirement       3859
Rank                    0
dtype: int64

## Xử lý thiếu ReleaseDate

`ReleaseDate` là thông tin quan trọng và số lượng thiếu không đáng kể, nên notebook loại bỏ các dòng thiếu ngày phát hành bằng `dropna`.

Cách xử lý này hợp lý khi tỷ lệ thiếu nhỏ và việc suy đoán ngày phát hành có thể gây sai lệch dữ liệu.


In [9]:
# Những ngày tháng không không hợp lệ sẽ được loại bỏ, với số lượng không đáng kể 
df.dropna(subset=['ReleaseDate'], inplace=True)

## Fill Developers và Publishers dựa vào nhau

Trong dữ liệu game, `Developers` và `Publishers` thường giống nhau hoặc có quan hệ gần. Nếu một trong hai cột thiếu, notebook dùng cột còn lại để điền.

Mục tiêu là giảm null nhưng vẫn giữ thông tin có ý nghĩa, thay vì fill bằng giá trị chung chung ngay từ đầu.


In [10]:
# Với cột dev và publisher, qua quá trình đọc hiểu dữ liệu, nhận thấy thường thì dev và publisher sẽ là một, hoặc có thể có nhiều dev nhưng chỉ có một publisher. 
df["Publishers"]= df["Publishers"].fillna(df["Developers"])
df['Developers'] = df['Developers'].fillna(df['Publishers'])

## Kiểm tra null sau bước fill Developers/Publishers

Cell này kiểm tra lại null để đánh giá bước xử lý trước có hiệu quả không.


In [11]:
df.isnull().sum()   

Appid                   0
Name                    0
Type                    0
ReleaseDate             0
Genres                411
Developers             11
Publishers             11
Description            52
price                   0
Thumbnail               0
Tags                 1610
ReviewScore             0
PositiveReview          0
NegativeReview          0
OsRequirement        2929
MemoryRequirement    3976
CpuRequirement       3851
Rank                    0
dtype: int64

## Loại bỏ game thiếu cả Developers và Publishers

Nếu cả `Developers` và `Publishers` đều thiếu, game đó thiếu thông tin nguồn gốc quan trọng.

Notebook loại bỏ các dòng này vì số lượng không đáng kể và có thể không phù hợp để đưa vào recommendation.


In [12]:
# Có thể thấy số null của cột developers và publishers là bằng nhau, với số lượng không đáng kể, những game không rõ nguồn gốc ta sẽ không đề xuất cho mọi người nên dev và pub cùng null ta sẽ bỏ 
df.dropna(subset=['Developers'], inplace=True)
df.isnull().sum()   

Appid                   0
Name                    0
Type                    0
ReleaseDate             0
Genres                406
Developers              0
Publishers              0
Description            50
price                   0
Thumbnail               0
Tags                 1603
ReviewScore             0
PositiveReview          0
NegativeReview          0
OsRequirement        2923
MemoryRequirement    3970
CpuRequirement       3844
Rank                    0
dtype: int64

## Kiểm tra các game thiếu Description

Cell này hiển thị các game chưa có mô tả để đánh giá mức độ ảnh hưởng trước khi fill.


In [13]:
display(df[df['Description'].isnull()])

,Appid,Name,Type,ReleaseDate,Genres,Developers,Publishers,Description,price,Thumbnail,Tags,ReviewScore,PositiveReview,NegativeReview,OsRequirement,MemoryRequirement,CpuRequirement,Rank
44,2357570,Overwatch®,game,"Aug 10, 2023","Action,Free To Play","Blizzard Entertainment, Inc.","Blizzard Entertainment, Inc.",NaN,0$,https://shared.akamai.steamstatic.com/store_it...,"Free to Play,Hero Shooter,FPS,Multiplayer,Team...",5,64,72,Windows® 10 64-bit (latest Service Pack),6 GB RAM,Intel® Core™ i3 or AMD Phenom™ X3 8650,45
107,1771300,Kingdom Come: Deliverance II,game,"Feb 4, 2025","Action,Adventure,RPG",Warhorse Studios,Deep Silver,NaN,59.99$,https://shared.akamai.steamstatic.com/store_it...,NaN,8,56046,4051,NaN,NaN,NaN,108
162,3932890,Escape from Tarkov,game,"Nov 15, 2025","Action,Massively Multiplayer,RPG,Simulation",Battlestate Games,Battlestate Games,NaN,49.99$,https://shared.akamai.steamstatic.com/store_it...,"Extraction Shooter,Shooter,Gun Customization,T...",5,5562,3689,Windows 10,16 GB RAM,AMD Ryzen 5 3600 or similar,163
166,2138330,Cyberpunk 2077: Phantom Liberty,dlc,"Sep 25, 2023",RPG,CD PROJEKT RED,CD PROJEKT RED,NaN,29.99$,https://shared.akamai.steamstatic.com/store_it...,NaN,8,9365,916,NaN,NaN,NaN,167
239,9900,Star Trek Online,game,"Jan 31, 2012","Massively Multiplayer,RPG,Free To Play",Cryptic Studios,Arc Games,NaN,0$,https://shared.akamai.steamstatic.com/store_it...,"Free to Play,Sci-fi,Massively Multiplayer,Spac...",6,642,198,NaN,NaN,NaN,240
907,3368610,Kingdom Come: Deliverance II Legacy of the Forge,dlc,"Sep 9, 2025","Action,Adventure,RPG",Warhorse Studios,Deep Silver,NaN,13.99$,https://shared.akamai.steamstatic.com/store_it...,NaN,5,488,225,NaN,NaN,NaN,908
916,1574820,Until Then,game,"Jun 25, 2024","Adventure,Casual,Indie",Polychroma Games,Maximum Entertainment,NaN,9.99$,https://shared.akamai.steamstatic.com/store_it...,"Visual Novel,Emotional,Pixel Graphics,Story Ri...",9,7600,162,Windows 7,4 GB RAM,Intel Core i5-4460 / AMD FX-8320,917
999,109600,Neverwinter,game,"Dec 5, 2013","Action,Adventure,Massively Multiplayer,RPG,Fre...",Cryptic Studios,Arc Games,NaN,0$,https://shared.akamai.steamstatic.com/store_it...,"Free to Play,MMORPG,RPG,Massively Multiplayer,...",5,75,33,Windows® 10/11 (64-bit),4 GB RAM,Core 2 Duo 2.2Ghz (or equivalent AMD CPU),1000
1072,3368620,Kingdom Come: Deliverance II Mysteria Ecclesiae,dlc,"Nov 11, 2025","Action,Adventure,RPG",Warhorse Studios,Deep Silver,NaN,13.99$,https://shared.akamai.steamstatic.com/store_it...,NaN,5,334,253,NaN,NaN,NaN,1073
1134,2067050,Squirrel with a Gun,game,"Aug 29, 2024","Action,Adventure,Indie,Simulation",Dee Dee Creations LLC,Maximum Entertainment,NaN,6.99$,https://shared.akamai.steamstatic.com/store_it...,"Action,Simulation,Third-Person Shooter,Sandbox...",8,2127,333,Windows 7 (64 bit),8 GB RAM,2.6 GHz,1135


## Fill Description thiếu

Các game thiếu mô tả vẫn có thể dùng các feature khác để recommendation, nên notebook không xóa chúng.

Thay vào đó, `Description` được fill bằng chuỗi `No description` để giữ dòng dữ liệu và tránh lỗi khi xử lý text sau này.


In [14]:
# Nhận thấy có khá nhiều game không có phần mô tả, tuy nhiên ta vẫn có thể sử dụng thông tin khác để khuyến nghị game nên ta sẽ fillna "No description" cho những game này
df['Description'] = df['Description'].fillna("No description")

In [15]:
df.isnull().sum()

Appid                   0
Name                    0
Type                    0
ReleaseDate             0
Genres                406
Developers              0
Publishers              0
Description             0
price                   0
Thumbnail               0
Tags                 1603
ReviewScore             0
PositiveReview          0
NegativeReview          0
OsRequirement        2923
MemoryRequirement    3970
CpuRequirement       3844
Rank                    0
dtype: int64

## Fill các yêu cầu hệ thống bị thiếu

Các cột như `OsRequirement`, `MemoryRequirement`, `CpuRequirement` nếu thiếu sẽ được xem như không có yêu cầu rõ ràng.

Notebook fill bằng `No requirement` để giúp các bước normalize OS/RAM/CPU sau này xử lý thống nhất.


In [16]:
# Với những thuộc tính yêu cầu về os, memory, cpu nếu NULL thì ta có thể coi nó là không có yêu cầu nên ta fillna bằng "No requirement" 
df['OsRequirement'] = df['OsRequirement'].fillna("No requirement")
df['MemoryRequirement'] = df['MemoryRequirement'].fillna("No requirement")  
df['CpuRequirement'] = df['CpuRequirement'].fillna("No requirement")  

In [17]:
df.isnull().sum()   

Appid                   0
Name                    0
Type                    0
ReleaseDate             0
Genres                406
Developers              0
Publishers              0
Description             0
price                   0
Thumbnail               0
Tags                 1603
ReviewScore             0
PositiveReview          0
NegativeReview          0
OsRequirement           0
MemoryRequirement       0
CpuRequirement          0
Rank                    0
dtype: int64

## Xử lý thiếu Genres và Tags

Cell này xử lý tags theo logic:

- Nếu `Genres` thiếu thì fill bằng chuỗi rỗng.
- Nếu `Tags` thiếu thì dùng `Genres` làm fallback.

Lý do: tags và genres đều mô tả nội dung game; dùng genres để thay tags giúp giảm mất thông tin.


In [18]:
df['Genres'] = df['Genres'].fillna('')
df['Tags'] = df['Tags'].fillna(df['Genres'])


In [19]:
df.isnull().sum()

Appid                0
Name                 0
Type                 0
ReleaseDate          0
Genres               0
Developers           0
Publishers           0
Description          0
price                0
Thumbnail            0
Tags                 0
ReviewScore          0
PositiveReview       0
NegativeReview       0
OsRequirement        0
MemoryRequirement    0
CpuRequirement       0
Rank                 0
dtype: int64

## Kiểm tra dòng thiếu cả Tags và Description

Cell này giúp phát hiện những dòng rất nghèo thông tin: thiếu tags và không có description.

Các dòng như vậy có thể gây khó khăn cho recommendation hoặc phân tích nội dung.


In [20]:
display(df[df['Tags'].isnull() & df['Description'] == "No description"])

,Appid,Name,Type,ReleaseDate,Genres,Developers,Publishers,Description,price,Thumbnail,Tags,ReviewScore,PositiveReview,NegativeReview,OsRequirement,MemoryRequirement,CpuRequirement,Rank


## Fill Tags còn thiếu bằng chuỗi rỗng

Sau khi đã dùng `Genres` làm fallback, nếu `Tags` vẫn thiếu thì fill bằng chuỗi rỗng.

Điều này giúp các hàm split/list sau này không bị lỗi do gặp `NaN`.


In [21]:
df['Tags'] = df['Tags'].fillna('')


In [22]:
df.isnull().sum()

Appid                0
Name                 0
Type                 0
ReleaseDate          0
Genres               0
Developers           0
Publishers           0
Description          0
price                0
Thumbnail            0
Tags                 0
ReviewScore          0
PositiveReview       0
NegativeReview       0
OsRequirement        0
MemoryRequirement    0
CpuRequirement       0
Rank                 0
dtype: int64

## Lưu dữ liệu sau bước cleaning

Cell này lưu DataFrame đã xử lý null và rank ra `data.csv`.

File này là output trung gian để dùng tiếp trong phần normalization.


In [23]:
df.to_csv('data.csv', index=False)

# Phần 2: PreprocessingNormalize(1).ipynb

*Nội dung được gộp từ file `PreprocessingNormalize(1).ipynb`.*


## Import lại thư viện cho phần normalization

Phần 2 có thể được chạy độc lập, nên notebook import lại các thư viện cần thiết.

Điều này giúp cell normalization vẫn chạy được kể cả khi người dùng bắt đầu từ phần 2.


In [24]:
import numpy as np 
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re

## Đọc lại dữ liệu cho phần normalization

Cell này đọc lại dữ liệu gốc hoặc dữ liệu đã clean tùy cách tổ chức file.

Mục tiêu của phần này là chuẩn hóa các feature phức tạp như price, OS, memory, CPU và tags.


In [25]:
# Đường dẫn Kaggle hiện tại của bạn. Nếu chạy theo project structure, dùng RAW_PATH.
kaggle_path = Path('/kaggle/input/datasets/newnguyn/steamgame/SteamGames.csv')

df = pd.read_csv(kaggle_path if kaggle_path.exists() else RAW_PATH)


## Xem cấu trúc DataFrame

`df.info()` cho biết số dòng, số cột, kiểu dữ liệu và số giá trị non-null của từng cột.

Đây là bước kiểm tra tổng quan trước khi normalize từng feature.


In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29931 entries, 0 to 29930
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Appid              29931 non-null  int64 
 1   Name               29931 non-null  object
 2   Type               29931 non-null  object
 3   ReleaseDate        29917 non-null  object
 4   Genres             29518 non-null  object
 5   Developers         29905 non-null  object
 6   Publishers         28886 non-null  object
 7   Description        29879 non-null  object
 8   price              29931 non-null  object
 9   Thumbnail          29931 non-null  object
 10  Tags               28317 non-null  object
 11  ReviewScore        29931 non-null  int64 
 12  PositiveReview     29931 non-null  int64 
 13  NegativeReview     29931 non-null  int64 
 14  OsRequirement      26994 non-null  object
 15  MemoryRequirement  25947 non-null  object
 16  CpuRequirement     26072 non-null  objec

## Kiểm tra null bằng `isnull()`

Cell này xem số lượng null từng cột. Kết quả dùng để xác nhận dữ liệu còn thiếu ở đâu trước khi normalize.


In [27]:
df.isnull().sum() 

Appid                   0
Name                    0
Type                    0
ReleaseDate            14
Genres                413
Developers             26
Publishers           1045
Description            52
price                   0
Thumbnail               0
Tags                 1614
ReviewScore             0
PositiveReview          0
NegativeReview          0
OsRequirement        2937
MemoryRequirement    3984
CpuRequirement       3859
Rank                    0
dtype: int64

## Kiểm tra null bằng `isna()`

`isna()` tương đương `isnull()` trong pandas. Cell này đóng vai trò kiểm tra lại để đảm bảo không bỏ sót missing values.


In [28]:
df.isna().sum()

Appid                   0
Name                    0
Type                    0
ReleaseDate            14
Genres                413
Developers             26
Publishers           1045
Description            52
price                   0
Thumbnail               0
Tags                 1614
ReviewScore             0
PositiveReview          0
NegativeReview          0
OsRequirement        2937
MemoryRequirement    3984
CpuRequirement       3859
Rank                    0
dtype: int64

## Chuẩn hóa cột price

Cell này chuyển `price` từ dạng text sang số:

- Xóa ký hiệu `$` và dấu phẩy.
- Chuyển `Free To Play` hoặc các giá trị free về `0`.
- Ép kiểu sang numeric.

Sau bước này, `price` có thể dùng cho phân tích, thống kê hoặc model ML.


In [29]:
df['price'] = (
    df['price']
    .astype(str)
    .str.strip()
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.replace('Free To Play', '0', regex=False)
    .str.replace('Free', '0', regex=False)
    .replace({'nan': np.nan, 'None': np.nan, '': np.nan})
)

df['price'] = pd.to_numeric(df['price'], errors='coerce')

display(df['price'].describe())
display(df['price'].isna().sum())


count    29931.000000
mean        11.833883
std         12.710452
min          0.000000
25%          3.990000
50%          8.990000
75%         14.990000
max        499.000000
Name: price, dtype: float64

np.int64(0)

## Kiểm tra phân phối OSRequirement gốc

Trước khi normalize OS, cell này xem các giá trị xuất hiện nhiều nhất trong `OsRequirement`.

Việc này giúp hiểu dữ liệu đang có những cách viết nào: Windows, Mac, Linux, no requirement, hoặc unknown.


In [30]:
display(df['OsRequirement'].value_counts())

OsRequirement
Windows 10                                                  3351
Windows 7                                                   2303
Windows 10 64-bit                                            626
Windows XP                                                   516
Windows 7+                                                   385
                                                            ... 
Windows XP/Vista/7/8 (32 bit or 64 bit)                        1
Windows 10 / 8.1 / 8 / 7 / Vista / XP (32-bit or 64-bit)       1
WINDOWS 8                                                      1
Win XP (Service Pack 3)                                        1
Windows 7, Win 8.1                                             1
Name: count, Length: 4027, dtype: int64

## Tách và chuẩn hóa hệ điều hành

Cell này định nghĩa hàm `extract_os()` để gom OS về các nhóm chuẩn:

- `windows`
- `mac`
- `linux`
- `no requirement`
- `unknown`

Hàm xử lý theo từ khóa vì dữ liệu yêu cầu hệ điều hành thường là text tự do và có nhiều cách viết.


In [31]:
# Chuẩn hóa OSRequirement về các nhóm chính: no requirement, windows, mac, linux, unknown.
def extract_os(x):
    x = str(x).lower()

    if (
        'no requirement' in x
        or 'any' == x.strip()
        or 'requires a 64-bit processor' in x
        or '64-bit processor and operating system' in x
    ):
        return ['no requirement']

    os_list = []

    if any(keyword in x for keyword in ['windows', 'win ', 'win7', 'win8', 'win10', 'win11', 'vista', 'xp']):
        os_list.append('windows')
    if any(keyword in x for keyword in ['mac', 'os x', 'macos']):
        os_list.append('mac')
    if any(keyword in x for keyword in ['linux', 'ubuntu', 'steamos']):
        os_list.append('linux')

    # Nếu hỗ trợ đủ 3 hệ điều hành chính thì coi như ít ràng buộc OS.
    if set(os_list) == {'windows', 'mac', 'linux'}:
        return ['no requirement']

    if not os_list:
        os_list.append('unknown')

    return os_list

df['OsNew'] = df['OsRequirement'].apply(extract_os)

## Kiểm tra kết quả OS sau normalize lần 1

Cell này đếm số lượng từng nhóm trong `OsNew` để xem hàm `extract_os()` đã phân loại tốt chưa.


In [32]:
display(df['OsNew'].value_counts())

OsNew
[windows]           26307
[unknown]            3497
[no requirement]      107
[windows, mac]         15
[linux]                 2
[windows, linux]        2
[mac]                   1
Name: count, dtype: int64

## Kiểm tra các OS còn unknown

Cell này lấy các dòng `OsNew` còn chứa `unknown` và xem lại giá trị gốc trong `OsRequirement`.

Mục tiêu là phát hiện các pattern còn thiếu để bổ sung rule xử lý.


In [33]:
df[df['OsNew'].apply(lambda x: 'unknown' in x)]['OsRequirement'].value_counts().head(20)

OsRequirement
10                  128
7                    71
7 / 8 / 8.1 / 10     33
Window 10            14
7/8.1/10             12
7,8,10,11             9
7+                    9
7/8/10                7
7 / 8 / 10            7
10+                   6
8                     6
7/8/10 32/64bit       5
TBA                   5
10/11                 5
Window 7              5
10, 11                5
7, 8, 10 (x64)        5
TBD                   5
-                     4
7 SP1+, 8, 10         4
Name: count, dtype: int64

## Sửa các trường hợp OS unknown còn lại

Sau khi kiểm tra, nhiều giá trị unknown thực ra là cách viết version Windows.

Cell này gom các trường hợp còn lại về `windows` nếu hợp lý, giúp giảm nhóm unknown.


In [34]:
# Với các dòng còn unknown, đa số là cách viết tắt version Windows nên gom về windows.
# Lưu ý: bước này chỉ chạy sau khi extract_os đã xử lý các case no requirement/mac/linux rõ ràng.
df['OsNew'] = df['OsNew'].apply(lambda x: ['windows'] if 'unknown' in x else x)
display(df['OsNew'].value_counts())

OsNew
[windows]           29804
[no requirement]      107
[windows, mac]         15
[linux]                 2
[windows, linux]        2
[mac]                   1
Name: count, dtype: int64

In [35]:
display(df['OsNew'].value_counts())

OsNew
[windows]           29804
[no requirement]      107
[windows, mac]         15
[linux]                 2
[windows, linux]        2
[mac]                   1
Name: count, dtype: int64

## Thay cột OS gốc bằng cột đã chuẩn hóa

Sau khi kiểm tra hợp lệ, notebook gán `OsRequirement = OsNew` và xóa cột tạm `OsNew`.

Từ đây trở đi, `OsRequirement` là version đã clean.


In [36]:
# Bây giờ ta đã tạo ra nhóm dễ xử lí hơn với 5 nhóm chính, giờ ta sẽ thay vào cột os requirement để tiện cho việc mã hóa sau này
df['OsRequirement'] = df['OsNew']
df.drop(columns=['OsNew'], inplace=True)

In [37]:
display(df['OsRequirement'].value_counts())

OsRequirement
[windows]           29804
[no requirement]      107
[windows, mac]         15
[linux]                 2
[windows, linux]        2
[mac]                   1
Name: count, dtype: int64

## Kiểm tra MemoryRequirement gốc

Cell này xem các giá trị phổ biến của RAM requirement trước khi viết hàm normalize.

Dữ liệu RAM thường có nhiều format như MB, GB, hoặc text mô tả.


In [38]:
# Tiếp tục là xử lí về ram .
display(df['MemoryRequirement'].value_counts())

MemoryRequirement
4 GB RAM                                                                                                                                                                                                                                                7428
8 GB RAM                                                                                                                                                                                                                                                5868
2 GB RAM                                                                                                                                                                                                                                                4307
1 GB RAM                                                                                                                                                                                                                       

## Chuẩn hóa yêu cầu RAM

Cell này định nghĩa hàm `normalize_memory()` để chuyển RAM về dạng số, thường là GB.

Hàm xử lý các trường hợp:

- `No requirement` → 0
- Giá trị theo MB → đổi sang GB
- Giá trị theo GB → giữ theo GB
- Không parse được → `NaN`

Kết quả được lưu vào cột tạm `NewMemoryRequirement`.


In [39]:
def normalize_memory(mem):
    if pd.isna(mem):
        return np.nan

    mem = str(mem).lower()

    if 'no requirement' in mem:
        return 0

    matches = re.findall(r'(\d+(?:\.\d+)?)\s*(gb|gigs|gigabyte|gigabytes|mb|m)', mem)
    values = []

    for num, unit in matches:
        num = float(num)
        if unit in ['gb', 'gigs', 'gigabyte', 'gigabytes']:
            values.append(num * 1024)
        else:
            values.append(num)

    return max(values) if values else np.nan

df['NewMemoryRequirement'] = df['MemoryRequirement'].apply(normalize_memory)

display(df['NewMemoryRequirement'].value_counts(dropna=False).head(20))


NewMemoryRequirement
4096.0     7553
8192.0     5913
2048.0     4658
NaN        4013
1024.0     2757
512.0      1341
16384.0     905
6144.0      760
256.0       440
3072.0      239
12288.0     203
4.0         152
128.0       142
500.0       102
8.0          88
2.0          60
64.0         54
5120.0       38
1.0          34
1536.0       32
Name: count, dtype: int64

## Kiểm tra kết quả RAM sau normalize

Cell này xem phân phối `NewMemoryRequirement` để kiểm tra có giá trị bất thường hay không.


In [40]:
display(df['NewMemoryRequirement'].value_counts())

NewMemoryRequirement
4096.0    7553
8192.0    5913
2048.0    4658
1024.0    2757
512.0     1341
          ... 
360.0        1
6000.0       1
1234.0       1
580.0        1
0.0          1
Name: count, Length: 103, dtype: int64

## Thay cột MemoryRequirement gốc

Sau khi kiểm tra, notebook thay `MemoryRequirement` bằng giá trị đã normalize.

Điều này giúp feature RAM trở thành dạng số có quy tắc, phù hợp cho model.


In [41]:
#Vậy ta đã biến feature này về dạng có quy tắc hơn rồi, sau quá trình kiểm tra cũng mang tính hợp lệ nên ta sẽ thay vào cột chính luôn 
df['MemoryRequirement'] = df['NewMemoryRequirement']
df.drop(columns=['NewMemoryRequirement'], inplace=True)

## Kiểm tra CpuRequirement gốc

Cell này xem các giá trị phổ biến trong yêu cầu CPU.

CPU là cột khó normalize vì text rất đa dạng, có thể chứa tên chip, GHz, số core hoặc mô tả chung.


In [42]:
display(df['CpuRequirement'].value_counts())

CpuRequirement
Intel Core i5                                                322
Intel Core i3                                                272
Intel Core2 Duo or better                                    266
2 GHz                                                        232
2.0 GHz                                                      181
                                                            ... 
Intel Core i3 (4th Generation)                                 1
Intel® Core™ 2 Duo @ 3.0GHz or AMD Athlon™ 64 X2 @ 3.0GHz      1
Intel Core i5-9400F                                            1
Intel® Core i3-3250/AMD FX-4350                                1
Intel Core i5-6600 / AMD Ryzen 5 2600                          1
Name: count, Length: 8430, dtype: int64

## Trích xuất thông tin CPU

Cell này định nghĩa logic để tách CPU thành các feature dễ dùng hơn:

- `CPU_req`: nhóm yêu cầu chung.
- `CPU_GHz`: tốc độ GHz nếu có.
- `CPU_tier`: mức CPU ước lượng theo keyword.

Cách làm này biến text CPU phức tạp thành nhiều cột có cấu trúc hơn.


In [43]:
#có thể thấy có rất nhiều giá trị khác nhau về yêu cầu của game và cpu, nhưng ta có thể thấy rằng vẫn có các nhóm từ khóa xuất hiện, nên ta sẽ chia thành 3 nhóm yêu cầu chính là cpu_tier, CPU_Ghz, Cpu_requirement
import re
import numpy as np
import pandas as pd

def extract_cpu_info(x):
    x = str(x).lower()
    x = x.replace(',', '.')
    
    cpu_tier = 0
    cpu_ghz = np.nan
    cpu_requirement = 1
    
    # Với trường hợp không yêu cầu về cpu 
    if 'no requirement' in x:
        cpu_requirement = 0
        cpu_ghz = 0 
        cpu_tier = 0
        return pd.Series([cpu_requirement, cpu_ghz, cpu_tier])
    
    # tách thông tin về GHz nếu có
    ghz_match = re.search(r'(\d+(\.\d+)?)\s*ghz', x)
    if ghz_match:
        cpu_ghz = float(ghz_match.group(1))

    # chia cấp bậc cho các loại cpu dựa trên từ khóa xuất hiện
    tiers = []
    if any(k in x for k in ['i3', 'ryzen 3']):
        tiers.append(1)
    if any(k in x for k in ['i5', 'ryzen 5']):
        tiers.append(2)
    if any(k in x for k in ['i7', 'ryzen 7']):
        tiers.append(3)
    if any(k in x for k in ['i9', 'ryzen 9']):
        tiers.append(4) 
    if any(k in x for k in ['core2', 'duo', 'pentium', 'dual core', 'celeron']):
        tiers.append(1)
    if any(k in x for k in ['any', 'almost any', 'multi-core']):
        tiers.append(1)
    if 'last' in x and 'year' in x:
        tiers.append(2)
    if len(tiers) > 0:
        cpu_tier = min(tiers)
    else:
        cpu_tier = 0
    
    return pd.Series([cpu_requirement, cpu_ghz, cpu_tier])

## Áp dụng hàm trích xuất CPU

Cell này chạy hàm `extract_cpu_info()` cho từng dòng trong `CpuRequirement` và tạo 3 cột mới:

`CPU_req`, `CPU_GHz`, `CPU_tier`.


In [44]:
df[['CPU_req', 'CPU_GHz', 'CPU_tier']] = df['CpuRequirement'].apply(extract_cpu_info)


## Kiểm tra phân phối các cột CPU mới

Cell này hiển thị số lượng từng giá trị trong `CPU_req`, `CPU_GHz`, và `CPU_tier`.

Mục tiêu là kiểm tra rule CPU có tạo ra nhóm hợp lý không.


In [45]:
display(df['CPU_req'].value_counts())
display(df['CPU_GHz'].value_counts())
display(df['CPU_tier'].value_counts())

CPU_req
1.0    29930
0.0        1
Name: count, dtype: int64

CPU_GHz
2.00    3296
2.40     947
1.00     848
3.00     832
2.50     774
        ... 
3.17       1
3.07       1
0.50       1
1.85       1
0.80       1
Name: count, Length: 71, dtype: int64

CPU_tier
1.0    12121
0.0    11178
2.0     6129
3.0      500
4.0        3
Name: count, dtype: int64

In [46]:
display(df['CPU_GHz'].value_counts())

CPU_GHz
2.00    3296
2.40     947
1.00     848
3.00     832
2.50     774
        ... 
3.17       1
3.07       1
0.50       1
1.85       1
0.80       1
Name: count, Length: 71, dtype: int64

In [47]:
df.isnull().sum()

Appid                    0
Name                     0
Type                     0
ReleaseDate             14
Genres                 413
Developers              26
Publishers            1045
Description             52
price                    0
Thumbnail                0
Tags                  1614
ReviewScore              0
PositiveReview           0
NegativeReview           0
OsRequirement            0
MemoryRequirement     4013
CpuRequirement        3859
Rank                     0
CPU_req                  0
CPU_GHz              17916
CPU_tier                 0
dtype: int64

## Fill giá trị thiếu trong CPU_GHz

Nhiều yêu cầu CPU không ghi rõ GHz, nên `CPU_GHz` có thể bị thiếu.

Notebook fill các giá trị thiếu bằng một giá trị đại diện phù hợp, thường dựa vào thống kê hoặc giả định yêu cầu trung bình.


In [48]:
#Với thành phần GHz có rất nhiều giá trị nan, là do trong feature yêu cầu về cpu có rất nhiều cách viết và đa số thì không sở hữu thông tin về GHz, vậy giờ với những dữ liệu nan, ta sẽ lấy giá trị median của các giá trị GHz đã có để thay thế, vì median sẽ không bị ảnh hưởng bởi các giá trị ngoại lai, và cũng sẽ phản ánh được xu hướng trung tâm của dữ liệu hơn là mean
med_ghz = df.groupby('CPU_tier')['CPU_GHz'].median()
def fill_ghz(row):
    if pd.isna(row['CPU_GHz']):
        return med_ghz[row['CPU_tier']]
    return row['CPU_GHz']
df['CPU_GHz'] = df.apply(fill_ghz, axis=1)

In [49]:
df.isnull().sum()

Appid                   0
Name                    0
Type                    0
ReleaseDate            14
Genres                413
Developers             26
Publishers           1045
Description            52
price                   0
Thumbnail               0
Tags                 1614
ReviewScore             0
PositiveReview          0
NegativeReview          0
OsRequirement           0
MemoryRequirement    4013
CpuRequirement       3859
Rank                    0
CPU_req                 0
CPU_GHz                 0
CPU_tier                0
dtype: int64

## Xóa cột CpuRequirement gốc

Sau khi đã tách CPU thành các cột có cấu trúc hơn, cột text gốc `CpuRequirement` được xóa để tránh dư thừa.


In [50]:
#Sau khi tách xong ta chỉ cần bỏ cột CpuRequirement
df.drop(columns=['CpuRequirement'], inplace=True)

## Tags



## Tách Genres/Tags thành list

Cell này định nghĩa `split_items()` để chuyển chuỗi tags/genres phân tách bằng dấu phẩy thành list lowercase.

Việc chuyển sang list giúp dễ map tag, đếm tần suất và lọc tags sau này.


In [51]:
def split_items(x):
    if pd.isna(x):
        return []

    items = []
    for item in str(x).split(','):
        item = item.strip().lower()
        if item and item not in items:
            items.append(item)
    return items

df['Tag_list_raw'] = df['Tags'].apply(split_items)
df['Genre_list'] = df['Genres'].apply(split_items)

df['num_tags_raw'] = df['Tag_list_raw'].apply(len)

display(df['num_tags_raw'].describe())
display(df['Tag_list_raw'].explode().dropna().value_counts().head(30))


count    29931.000000
mean        14.073669
std          7.388251
min          0.000000
25%          7.000000
50%         19.000000
75%         20.000000
max         20.000000
Name: num_tags_raw, dtype: float64

Tag_list_raw
singleplayer          16920
indie                 13052
adventure             11506
action                11285
casual                 9572
simulation             8591
2d                     8183
strategy               6916
atmospheric            6491
rpg                    6146
story rich             5943
3d                     5282
multiplayer            5179
exploration            5090
cute                   5022
puzzle                 4807
colorful               4552
pixel graphics         4439
first-person           4386
funny                  4068
fantasy                4034
anime                  3802
relaxing               3518
horror                 3363
female protagonist     3314
family friendly        3282
retro                  3007
co-op                  2899
sci-fi                 2844
open world             2800
Name: count, dtype: int64

## Rule-based tag mapping

Cell này tạo dictionary `tag_mapping` để gom các cách viết tương đương về một tag chuẩn.

Ví dụ:

- `role-playing`, `role playing` → `rpg`
- `rogue-lite`, `rogue-like` → `roguelike`

Đây là bước mapping thủ công trước khi dùng LLM, giúp giảm số lượng tag cần gọi Qwen.


In [52]:
tag_mapping = {
    'role-playing': 'rpg',
    'role playing': 'rpg',
    'rogue-lite': 'roguelike',
    'rogue like': 'roguelike',
    'rogue-like': 'roguelike',
    'action-adventure': 'action adventure',
    'hack & slash': 'hack and slash',
    'hack n slash': 'hack and slash',
    'point & click': 'point and click',
    'point-and-click': 'point and click',
    'free to play': 'free-to-play',
    'f2p': 'free-to-play',
    'coop': 'co-op',
    'multi-player': 'multiplayer',
    'single-player': 'singleplayer',
}

def map_tag_list(tags):
    mapped = []
    for tag in tags:
        tag = tag_mapping.get(tag, tag).strip()
        if tag and tag not in mapped:
            mapped.append(tag)
    return mapped

df['Tag_list_mapped'] = df['Tag_list_raw'].apply(map_tag_list)


## LLM-assisted tag normalization bằng Qwen 7B 4-bit — bản tối ưu

Phần này chuẩn hóa các Steam tags hiếm/không nhất quán về danh sách `canonical_tags`.

Thiết kế mới tập trung vào 4 điểm:

1. **Few-shot prompting** để Qwen hiểu rõ kiểu mapping mong muốn.
2. **Parse JSON chắc hơn** kể cả khi model lỡ trả về markdown/code fence.
3. **Validate canonical tags bằng lowercase** để tránh lỗi `Valid mappings = 0` do khác hoa/thường.
4. **Retry + debug log** để dễ kiểm tra batch lỗi hoặc output rỗng.

> Trên Kaggle cần bật GPU và Internet trước khi chạy cell load model.



### 1. Cài thư viện cần thiết

Chạy cell này trên Kaggle nếu môi trường chưa có `transformers`, `accelerate`, `bitsandbytes`, hoặc `sentencepiece`.
Nếu Kaggle yêu cầu restart kernel sau khi cài, hãy restart rồi chạy lại notebook từ đầu.



## Cài thư viện cho Qwen 7B 4-bit

Cell này cài các thư viện cần cho inference LLM:

- `transformers`: load tokenizer/model.
- `accelerate`: hỗ trợ device placement.
- `bitsandbytes`: chạy model 4-bit để giảm VRAM.
- `sentencepiece`: tokenizer dependency cho một số model.

Trên Kaggle, nếu thư viện đã có sẵn thì cell này chạy nhanh hoặc có thể bỏ qua.


In [53]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 87.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 102.3 MB/s eta 0:00:0000:01


### 2. Cấu hình LLM mapping

Các tham số bên dưới kiểm soát model, số canonical tags, batch size và vị trí lưu file mapping trung gian.



## Khai báo cấu hình mapping bằng LLM

Cell này chứa các tham số quan trọng:

- `LLM_MODEL_NAME`: model Qwen dùng để map tag.
- `MAX_CANONICAL_TAGS`: số tag phổ biến nhất được xem là bộ tag chuẩn.
- `MIN_TAG_FREQ_FOR_LLM`: chỉ đưa tag đủ tần suất vào LLM để tránh map noise.
- `LLM_BATCH_SIZE`: số tag xử lý trong mỗi lần gọi model.
- `TAG_MAPPING_PATH`: nơi lưu mapping để tái sử dụng.

Thay đổi các biến này sẽ ảnh hưởng trực tiếp đến tốc độ, chất lượng mapping và số lượng tag cuối cùng.


In [54]:
LLM_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# Số tag phổ biến nhất được dùng làm vocabulary chuẩn cho LLM chọn.
MAX_CANONICAL_TAGS = 200

# Chỉ đưa tag hiếm vào LLM nếu tag đó xuất hiện ít nhất số lần này.
MIN_RARE_TAG_FREQ = 2

# Batch nhỏ giúp Qwen trả JSON ổn định hơn.
LLM_BATCH_SIZE = 8

# Retry nếu model trả JSON rỗng hoặc lỗi parse.
LLM_MAX_RETRIES = 2

TAG_MAPPING_PATH = INTERIM_DIR / "tag_mapping_llm_qwen7b.csv"
TAGS_FOR_LLM_PATH = INTERIM_DIR / "tags_for_llm.csv"
VALIDATION_SAMPLE_PATH = INTERIM_DIR / "tag_mapping_llm_validation_sample.csv"

print("Model:", LLM_MODEL_NAME)
print("Batch size:", LLM_BATCH_SIZE)
print("Max retries:", LLM_MAX_RETRIES)


Model: Qwen/Qwen2.5-7B-Instruct
Batch size: 8
Max retries: 2


### 3. Chuẩn bị `canonical_tags` và `tags_for_llm`

- `canonical_tags`: top tags phổ biến nhất sau rule-based mapping.
- `tags_for_llm`: các tags chưa thuộc canonical vocabulary nhưng vẫn xuất hiện đủ nhiều để đáng chuẩn hóa.



## Chuẩn bị canonical tags và tags cần LLM xử lý

Cell này tạo 2 danh sách chính:

- `canonical_tags`: top tags phổ biến nhất sau bước rule-based mapping, dùng làm nhãn chuẩn cho Qwen chọn.
- `tags_for_llm`: các tag chưa nằm trong canonical list và cần model hỗ trợ mapping.

Ý tưởng là không để Qwen tự nghĩ tag mới, mà chỉ chọn từ danh sách canonical có sẵn.


In [55]:
base_tag_counts = df["Tag_list_mapped"].explode().dropna().value_counts()

canonical_tags = [
    str(tag).strip().lower()
    for tag in base_tag_counts.head(MAX_CANONICAL_TAGS).index.tolist()
]

all_mapped_tags = sorted(
    str(tag).strip().lower()
    for tag in df["Tag_list_mapped"].explode().dropna().unique()
)

tags_for_llm = [
    tag for tag in all_mapped_tags
    if tag not in canonical_tags and base_tag_counts.get(tag, 0) >= MIN_RARE_TAG_FREQ
]

pd.DataFrame({"tag": tags_for_llm}).to_csv(TAGS_FOR_LLM_PATH, index=False)

print("Canonical tags:", len(canonical_tags))
print("Tags for LLM:", len(tags_for_llm))
print("Saved tags_for_llm:", TAGS_FOR_LLM_PATH)

display(pd.DataFrame({"tag": tags_for_llm}).head(20))


Canonical tags: 200
Tags for LLM: 246
Saved tags_for_llm: /kaggle/working/data/interim/tags_for_llm.csv


,tag
0,2d fighter
1,360 video
2,3d fighter
3,3d vision
4,4x
5,6dof
6,8-bit music
7,action rts
8,addictive
9,agriculture


### 4. Load Qwen 7B Instruct ở 4-bit

Cell này tải model bằng `BitsAndBytesConfig(load_in_4bit=True)` để giảm VRAM trên Kaggle GPU.



## Load Qwen 7B Instruct ở 4-bit

Cell này load tokenizer và model Qwen với `BitsAndBytesConfig(load_in_4bit=True)`.

Lý do dùng 4-bit:

- Giảm VRAM đáng kể.
- Phù hợp môi trường Kaggle GPU.
- Vẫn đủ tốt cho tác vụ mapping tag ngắn.

Nếu bị lỗi thiếu GPU hoặc thiếu bitsandbytes, hãy kiểm tra runtime Kaggle đã bật GPU chưa.


In [56]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL_NAME,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.eval()

print("Loaded:", LLM_MODEL_NAME)


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-7B-Instruct


### 5. Helper functions: normalize, parse JSON, build prompt

Các hàm này giúp:

- chuẩn hóa text về lowercase;
- lấy JSON object từ output của Qwen;
- tạo prompt có few-shot example để model trả mapping ổn định hơn.



## Helper functions cho LLM output

Cell này gồm các hàm phụ trợ:

- `normalize_text()`: chuẩn hóa string để so sánh tag không bị lệch hoa/thường/khoảng trắng.
- `extract_json()`: lấy JSON object từ output của model, kể cả khi model lỡ thêm markdown fence.
- `build_qwen_prompt()`: tạo prompt có few-shot để Qwen hiểu format mapping.

Đây là phần giúp giảm lỗi `Valid mappings: 0` do parse JSON hoặc validate sai canonical tag.


In [57]:
import json
import re
import time
from pathlib import Path


def normalize_text(value):
    """Normalize tag text for matching and validation."""
    if value is None:
        return ""
    return str(value).strip().lower()


def extract_json(text):
    """
    Extract the first JSON object from LLM output.
    Returns {} if parsing fails.
    """
    if text is None:
        return {}

    text = str(text).strip()

    # Remove common markdown fences if Qwen accidentally returns them.
    text = re.sub(r"^```(?:json)?", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"```$", "", text).strip()

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return {}

    try:
        parsed = json.loads(match.group(0))
        return parsed if isinstance(parsed, dict) else {}
    except Exception:
        return {}


def build_qwen_prompt(batch_tags, canonical_tags):
    """Build few-shot prompt for Steam tag normalization."""
    return f"""
You are normalizing Steam game tags.

Task:
Map each input tag to exactly one allowed canonical tag if the meaning is clearly similar.
If no good match exists, return null.

Allowed canonical tags:
{json.dumps(canonical_tags, ensure_ascii=False)}

Rules:
- Use only tags from the allowed canonical tags.
- Do not invent new tags.
- Return JSON only.
- Do not return markdown.
- Do not explain.
- Keep the original input tag as the JSON key.
- The JSON value must be either one canonical tag or null.
- Prefer semantic meaning over exact spelling.
- If the input tag is too vague, too specific, unrelated, or not clearly equivalent, return null.

Few-shot example:

Allowed canonical tags:
["action", "adventure", "rpg", "strategy", "simulation", "casual", "sports", "racing", "horror", "puzzle", "multiplayer", "singleplayer", "open world", "survival", "shooter"]

Input tags:
["fps", "first-person shooter", "driving", "scary", "co-op", "football", "jrpg", "relaxing", "sandbox", "base building", "turn-based tactics"]

Output:
{{
  "fps": "shooter",
  "first-person shooter": "shooter",
  "driving": "racing",
  "scary": "horror",
  "co-op": "multiplayer",
  "football": "sports",
  "jrpg": "rpg",
  "relaxing": "casual",
  "sandbox": "open world",
  "base building": null,
  "turn-based tactics": "strategy"
}}

Now normalize these input tags.

Input tags:
{json.dumps(batch_tags, ensure_ascii=False)}

Return JSON only:
""".strip()


### 6. Hàm gọi Qwen và chạy mapping theo batch

Điểm quan trọng nhất trong cell này là decode **chỉ phần token mới sinh ra**:

```python
output[0][inputs.input_ids.shape[-1]:]
```

Cách này an toàn hơn `decoded[len(text):]` vì chuỗi trước tokenize và sau decode không luôn khớp 100%.



## Gọi Qwen và chạy mapping theo batch

Cell này là lõi của LLM mapping:

- `call_qwen_tag_mapping()`: gọi model cho một batch tags và parse JSON trả về.
- `run_qwen_mapping()`: chia tags thành nhiều batch, retry nếu output rỗng, validate mapping và gom kết quả.

Điểm quan trọng: notebook decode **chỉ phần token mới sinh ra** thay vì cắt chuỗi bằng `len(text)`, giúp tránh lỗi output rỗng hoặc sai vị trí.


In [58]:
def call_qwen_tag_mapping(batch_tags, canonical_tags, max_new_tokens=768, debug=False):
    """Call Qwen and return parsed mapping dict for one batch."""
    prompt = build_qwen_prompt(batch_tags, canonical_tags)

    messages = [
        {
            "role": "system",
            "content": "You are a strict JSON generator. Return only valid JSON and nothing else.",
        },
        {
            "role": "user",
            "content": prompt,
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output[0][inputs.input_ids.shape[-1]:]
    generated = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    if debug:
        print("=" * 80)
        print("BATCH:", batch_tags)
        print("RAW GENERATED:")
        print(repr(generated[:2000]))

    return extract_json(generated)


def run_qwen_mapping(
    tags_for_llm,
    canonical_tags,
    batch_size=8,
    sleep_time=0.2,
    max_retries=2,
    debug=False,
):
    """
    Run Qwen mapping over all tags.

    Returns:
        dict: {raw_tag_lower: mapped_canonical_tag_lower_or_None}
    """
    tags_for_llm = [normalize_text(tag) for tag in tags_for_llm if normalize_text(tag)]
    canonical_tags = [normalize_text(tag) for tag in canonical_tags if normalize_text(tag)]

    canonical_norm_set = set(canonical_tags)
    mapping = {}

    total = len(tags_for_llm)

    for start in range(0, total, batch_size):
        batch = tags_for_llm[start:start + batch_size]
        batch_set = set(batch)
        result = {}

        for attempt in range(max_retries + 1):
            try:
                result = call_qwen_tag_mapping(
                    batch_tags=batch,
                    canonical_tags=canonical_tags,
                    debug=debug and attempt == 0,
                )

                if result:
                    break

                print(
                    f"Empty JSON batch {start // batch_size + 1}, "
                    f"retry {attempt + 1}/{max_retries}"
                )

            except Exception as exc:
                print(
                    f"Failed batch {start // batch_size + 1}, "
                    f"retry {attempt + 1}/{max_retries}: {exc}"
                )

            time.sleep(sleep_time)

        # Make sure every input tag has an output key.
        for raw_tag in batch:
            mapping.setdefault(raw_tag, None)

        # Validate and keep only canonical values.
        for raw_tag, mapped_tag in result.items():
            raw_tag_norm = normalize_text(raw_tag)

            # Ignore hallucinated keys not in current batch.
            if raw_tag_norm not in batch_set:
                continue

            if mapped_tag is None:
                mapping[raw_tag_norm] = None
                continue

            mapped_tag_norm = normalize_text(mapped_tag)

            if mapped_tag_norm in canonical_norm_set:
                mapping[raw_tag_norm] = mapped_tag_norm
            else:
                mapping[raw_tag_norm] = None
                if debug:
                    print(f"Rejected mapping: {raw_tag} -> {mapped_tag}")

        print(f"Done {min(start + batch_size, total)}/{total}")
        time.sleep(sleep_time)

    return mapping


### 7. Chạy Qwen mapping hoặc load file đã có

Nếu file `tag_mapping_llm_qwen7b.csv` đã tồn tại thì notebook sẽ load lại để tránh gọi model nhiều lần.
Nếu muốn chạy lại từ đầu, xóa file này trong `/kaggle/working/data/interim/` rồi chạy lại cell.



## Load mapping cũ hoặc chạy Qwen mapping mới

Cell này giúp tiết kiệm thời gian:

- Nếu file mapping CSV đã tồn tại, notebook load lại để không phải gọi Qwen lần nữa.
- Nếu chưa có file, notebook chạy Qwen để tạo mapping mới rồi lưu ra CSV.

Sau khi chạy, notebook in ra:

- tổng số mapping,
- số mapping hợp lệ,
- một vài dòng sample để kiểm tra nhanh.


In [59]:
def load_existing_mapping(path):
    """Load mapping CSV and normalize None/null/nan safely."""
    mapping_df = pd.read_csv(path)

    loaded_mapping = {}
    for raw_tag, mapped_tag in zip(mapping_df["raw_tag"], mapping_df["mapped_tag"]):
        raw_tag_norm = normalize_text(raw_tag)

        if pd.isna(mapped_tag) or normalize_text(mapped_tag) in {"", "none", "nan", "null"}:
            loaded_mapping[raw_tag_norm] = None
        else:
            loaded_mapping[raw_tag_norm] = normalize_text(mapped_tag)

    return loaded_mapping


def save_mapping(mapping, path):
    """Save mapping dict to CSV."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    mapping_df = pd.DataFrame(
        [{"raw_tag": raw_tag, "mapped_tag": mapped_tag} for raw_tag, mapped_tag in mapping.items()]
    )

    mapping_df.to_csv(path, index=False)
    return mapping_df


if TAG_MAPPING_PATH.exists():
    tag_mapping_llm = load_existing_mapping(TAG_MAPPING_PATH)
    print("Loaded existing LLM mapping:", TAG_MAPPING_PATH)
else:
    tag_mapping_llm = run_qwen_mapping(
        tags_for_llm=tags_for_llm,
        canonical_tags=canonical_tags,
        batch_size=LLM_BATCH_SIZE,
        max_retries=LLM_MAX_RETRIES,
        debug=False,
    )
    save_mapping(tag_mapping_llm, TAG_MAPPING_PATH)
    print("Saved LLM mapping:", TAG_MAPPING_PATH)

print("LLM mapping size:", len(tag_mapping_llm))
print("Valid mappings:", sum(value is not None for value in tag_mapping_llm.values()))

display(pd.DataFrame(list(tag_mapping_llm.items()), columns=["raw_tag", "mapped_tag"]).head(20))


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Done 8/246
Done 16/246
Done 24/246
Done 32/246
Done 40/246
Done 48/246
Done 56/246
Done 64/246
Done 72/246
Done 80/246
Done 88/246
Done 96/246
Done 104/246
Done 112/246
Done 120/246
Done 128/246
Done 136/246
Done 144/246
Done 152/246
Done 160/246
Done 168/246
Done 176/246
Done 184/246
Done 192/246
Done 200/246
Done 208/246
Done 216/246
Done 224/246
Done 232/246
Done 240/246
Done 246/246
Saved LLM mapping: /kaggle/working/data/interim/tag_mapping_llm_qwen7b.csv
LLM mapping size: 246
Valid mappings: 69


,raw_tag,mapped_tag
0,2d fighter,action
1,360 video,None
2,3d fighter,action
3,3d vision,None
4,4x,strategy
5,6dof,None
6,8-bit music,None
7,action rts,rts
8,addictive,None
9,agriculture,None


### 8. Áp dụng LLM mapping và lọc tag cuối

Tag được map về `None/null` sẽ bị bỏ. Tag không có trong mapping được giữ nguyên.
Sau đó notebook giữ các tag phổ biến nhất và giới hạn số tag mỗi game.



## Áp dụng LLM mapping vào tags cuối cùng

Cell này dùng `tag_mapping_llm` để chuẩn hóa danh sách tag của từng game.

Logic chính:

1. Tag nào map về canonical tag thì thay bằng canonical tag.
2. Tag nào map về `None` thì bỏ.
3. Tag không có trong mapping thì giữ nguyên.
4. Lọc theo tần suất và giới hạn số tag mỗi game.
5. Nếu game có quá ít tag, dùng `Genres` làm fallback.

Kết quả là cột tag cuối cùng sạch hơn, ít nhiễu hơn và phù hợp cho recommendation.


In [63]:
TAG_TOP_N = 200
MIN_TAG_FREQ = 5
MAX_TAGS_PER_GAME = 10
MIN_TAGS_PER_GAME = 3


def apply_llm_mapping(tags):
    mapped_tags = []

    for tag in tags:
        tag_norm = normalize_text(tag)
        mapped = tag_mapping_llm.get(tag_norm, tag_norm)

        if mapped is None or pd.isna(mapped):
            continue

        mapped_norm = normalize_text(mapped)

        if mapped_norm and mapped_norm not in mapped_tags:
            mapped_tags.append(mapped_norm)

    return mapped_tags


df["Tag_list_llm"] = df["Tag_list_mapped"].apply(apply_llm_mapping)

tag_counts = df["Tag_list_llm"].explode().dropna().value_counts()
valid_tags = set(tag_counts[tag_counts >= MIN_TAG_FREQ].head(TAG_TOP_N).index)


def keep_tags(tags):
    kept_tags = [tag for tag in tags if tag in valid_tags]
    kept_tags = sorted(kept_tags, key=lambda tag: tag_counts.get(tag, 0), reverse=True)
    return kept_tags[:MAX_TAGS_PER_GAME]


df["Tag_list_clean"] = df["Tag_list_llm"].apply(keep_tags)
df["num_tags_clean"] = df["Tag_list_clean"].apply(len)

print("Unique tags after LLM mapping:", df["Tag_list_llm"].explode().dropna().nunique())
print("Valid final tags:", len(valid_tags))
print("LLM mappings used:", sum(value is not None for value in tag_mapping_llm.values()))

# Games with too few tags after filtering. Useful for later inspection.
low_tag_rows = df[df["num_tags_clean"] < MIN_TAGS_PER_GAME]
print("Rows with fewer than MIN_TAGS_PER_GAME:", len(low_tag_rows))

display(df["num_tags_clean"].describe())
display(df["Tag_list_clean"].explode().dropna().value_counts().head(30))


Unique tags after LLM mapping: 200
Valid final tags: 200
LLM mappings used: 69
Rows with fewer than MIN_TAGS_PER_GAME: 4030


count    29931.000000
mean         7.980856
std          3.395266
min          0.000000
25%          6.000000
50%         10.000000
75%         10.000000
max         10.000000
Name: num_tags_clean, dtype: float64

Tag_list_clean
singleplayer          16920
indie                 13052
adventure             11506
action                11485
casual                 9572
simulation             8803
2d                     8183
strategy               6926
atmospheric            6501
rpg                    6299
story rich             5943
multiplayer            5375
3d                     5282
exploration            5089
cute                   5020
puzzle                 4902
colorful               4519
pixel graphics         4382
first-person           4342
funny                  3940
fantasy                3857
anime                  3601
horror                 3268
relaxing               3171
shooter                2897
female protagonist     2811
retro                  2682
family friendly        2643
co-op                  2561
sci-fi                 2533
Name: count, dtype: int64

### 9. Tạo validation sample cho LLM mapping

File này dùng để kiểm tra thủ công mapping của LLM.
Cột `human_check` có thể điền `correct`, `incorrect`, hoặc `uncertain`.



## Tạo file sample để kiểm tra thủ công LLM mapping

Cell này lấy một phần mapping hợp lệ và lưu thành CSV validation.

File validation có cột `human_check` để người đọc có thể đánh dấu:

- `correct`: mapping đúng.
- `incorrect`: mapping sai.
- `uncertain`: không chắc.

Bước này rất hữu ích vì LLM mapping nên được kiểm tra thủ công một phần trước khi dùng cho báo cáo hoặc model chính thức.


In [64]:
validation_sample = (
    pd.DataFrame(list(tag_mapping_llm.items()), columns=["raw_tag", "mapped_tag"])
    .dropna(subset=["mapped_tag"])
)

if len(validation_sample) > 0:
    validation_sample = (
        validation_sample
        .sample(min(50, len(validation_sample)), random_state=42)
        .assign(human_check="")
    )
else:
    validation_sample = pd.DataFrame(columns=["raw_tag", "mapped_tag", "human_check"])

validation_sample.to_csv(VALIDATION_SAMPLE_PATH, index=False)

print("Saved validation sample:", VALIDATION_SAMPLE_PATH)
display(validation_sample.head(20))


Saved validation sample: /kaggle/working/data/interim/tag_mapping_llm_validation_sample.csv


,raw_tag,mapped_tag,human_check
97,gothic,horror,
0,2d fighter,action,
175,roguevania,metroidvania,
11,ambient,atmospheric,
187,skating,sports,
78,extraction shooter,shooter,
41,character action game,action,
135,motorbike,racing,
159,pinball,arcade,
44,co-op campaign,multiplayer,
